In [ ]:
from pathlib import Path
import json
import pickle
import subprocess
import sys
import warnings
import mlflow

import numpy as np
import optuna
import pandas as pd
import scipy.stats as stats
from scipy.stats import loguniform
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
mlflow.set_experiment("Small_Data_Optimization")

In [ ]:
ARTIFACT_DIR = Path("artifacts")
DATASET_PATH = ARTIFACT_DIR / "openml_prepared_datasets.pkl"
SPLIT_PATH = ARTIFACT_DIR / "nested_cv_splits.pkl"
RESULTS_PATH = ARTIFACT_DIR / "baseline_optimizer_results.csv"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_EVALUATIONS = 200 
N_JOBS = -1
OPTUNA_CV_JOBS = -1

with DATASET_PATH.open("rb") as dataset_file:
    prepared_datasets = pickle.load(dataset_file)

with SPLIT_PATH.open("rb") as split_file:
    nested_cv_splits = pickle.load(split_file)

# C_VALUES = np.logspace(-3, 3, 25).tolist()
# GAMMA_VALUES = np.logspace(-4, 1, 40).tolist()
C_VALUES = np.logspace(-3, 3, 10).tolist()
GAMMA_VALUES = np.logspace(-4, 1, 20).tolist()

PARAM_GRID = {
    "xgb__n_estimators": [50, 100, 200],
    "xgb__learning_rate": [0.01, 0.1, 0.2],
    "xgb__max_depth": [3, 5, 7],
}

PARAM_DISTRIBUTIONS = {
    "xgb__n_estimators": stats.randint(50, 500),
    "xgb__learning_rate": loguniform(1e-3, 1.0),
    "xgb__max_depth": stats.randint(3, 15),
}

In [ ]:
def build_pipeline(random_state: int) -> Pipeline:
    return Pipeline([
        ("scaler", StandardScaler()),
        ("xgb", XGBClassifier(random_state=random_state, eval_metric="mlogloss", use_label_encoder=False)),
    ])

def build_relative_inner_cv(outer_train_idx: np.ndarray, inner_splits: list) -> list:
    position_by_index = {absolute_index: relative_index for relative_index, absolute_index in enumerate(outer_train_idx)}
    relative_splits = []

    for split in inner_splits:
        train_idx = np.array([position_by_index[index] for index in split["train_idx"]], dtype=int)
        val_idx = np.array([position_by_index[index] for index in split["val_idx"]], dtype=int)
        relative_splits.append((train_idx, val_idx))

    return relative_splits

def normalize_params(params: dict) -> dict:
    normalized = {}
    for key, value in params.items():
        if isinstance(value, np.generic):
            normalized[key] = value.item()
        else:
            normalized[key] = value
    return normalized

def evaluate_grid_search(X_train, y_train, X_test, y_test, inner_cv, random_state: int):
    search = GridSearchCV(
        estimator=build_pipeline(random_state),
        param_grid=PARAM_GRID,
        scoring="accuracy",
        cv=inner_cv,
        refit=True,
        n_jobs=N_JOBS,
    )
    search.fit(X_train, y_train)
    best_index = search.best_index_
    cv_mean = float(search.cv_results_["mean_test_score"][best_index])
    cv_std = float(search.cv_results_["std_test_score"][best_index])
    test_acc = float(search.best_estimator_.score(X_test, y_test))
    return search.best_estimator_, cv_mean, cv_std, test_acc, normalize_params(search.best_params_)

def evaluate_random_search(X_train, y_train, X_test, y_test, inner_cv, random_state: int):
    search = RandomizedSearchCV(
        estimator=build_pipeline(random_state),
        param_distributions=PARAM_DISTRIBUTIONS,
        n_iter=N_EVALUATIONS,
        scoring="accuracy",
        cv=inner_cv,
        random_state=random_state,
        refit=True,
        n_jobs=N_JOBS,
    )
    search.fit(X_train, y_train)
    best_index = search.best_index_
    cv_mean = float(search.cv_results_["mean_test_score"][best_index])
    cv_std = float(search.cv_results_["std_test_score"][best_index])
    test_acc = float(search.best_estimator_.score(X_test, y_test))
    return search.best_estimator_, cv_mean, cv_std, test_acc, normalize_params(search.best_params_)

def evaluate_optuna(X_train, y_train, X_test, y_test, inner_cv, random_state: int):
    history = []

    def objective(trial):
        params = {
            "xgb__n_estimators": trial.suggest_int("xgb__n_estimators", 50, 500),
            "xgb__learning_rate": trial.suggest_float("xgb__learning_rate", 1e-3, 1.0, log=True),
            "xgb__max_depth": trial.suggest_int("xgb__max_depth", 3, 15),
        }
        model = build_pipeline(random_state).set_params(**params)
        scores = cross_val_score(
            model, X_train, y_train, cv=inner_cv, scoring="accuracy", n_jobs=OPTUNA_CV_JOBS
        )
        return float(np.mean(scores))

    def optuna_callback(study, trial):
        history.append({
            "trial": trial.number + 1,       
            "trial_value": trial.value,  
            "best_value": study.best_value   
        })

    sampler = optuna.samplers.TPESampler(seed=random_state)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    
    study.optimize(objective, n_trials=N_EVALUATIONS, callbacks=[optuna_callback], show_progress_bar=False)

    best_params = normalize_params(study.best_params)
    best_model = build_pipeline(random_state).set_params(**best_params)
    
    best_scores = cross_val_score(
        best_model, X_train, y_train, cv=inner_cv, scoring="accuracy", n_jobs=OPTUNA_CV_JOBS
    )
    cv_mean = float(np.mean(best_scores))
    cv_std = float(np.std(best_scores))
    test_acc = float(best_model.fit(X_train, y_train).score(X_test, y_test))

    history_df = pd.DataFrame(history)
    
    return best_model, cv_mean, cv_std, test_acc, best_params, history_df

def log_fold_metrics(dataset_name: str, method_name: str, outer_fold: int, cv_mean: float, cv_std: float, test_acc: float, best_params: dict):
    mlflow.log_param("dataset", dataset_name)
    mlflow.log_param("method", method_name)
    mlflow.log_param("outer_fold", outer_fold)
    mlflow.log_param("evaluation_budget", N_EVALUATIONS)
    mlflow.log_metric("cv_mean", cv_mean)
    mlflow.log_metric("cv_std", cv_std)
    mlflow.log_metric("test_acc", test_acc)
    mlflow.log_params({f"best_{key}": value for key, value in best_params.items()})

In [ ]:
results = []

for dataset_name, dataset_payload in prepared_datasets.items():
    X = dataset_payload["X"]
    y = dataset_payload["y"]
    outer_splits = nested_cv_splits[dataset_name]["outer_splits"]

    with mlflow.start_run(run_name=dataset_name):
        mlflow.log_param("n_outer_folds", len(outer_splits))
        mlflow.log_param("n_inner_folds", len(outer_splits[0]["inner_splits"]))
        mlflow.log_param("model", "XGBoost")

        for outer_fold_info in outer_splits:
            outer_fold = int(outer_fold_info["fold"])
            outer_train_idx = np.asarray(outer_fold_info["train_idx"], dtype=int)
            outer_test_idx = np.asarray(outer_fold_info["test_idx"], dtype=int)
            inner_cv = build_relative_inner_cv(outer_train_idx, outer_fold_info["inner_splits"])

            X_train = X.iloc[outer_train_idx]
            X_test = X.iloc[outer_test_idx]
            y_train = y[outer_train_idx]
            y_test = y[outer_test_idx]

            fold_seed = RANDOM_STATE + outer_fold
            method_runners = [
                ("grid_search", evaluate_grid_search),
                ("random_search", evaluate_random_search),
                ("optuna", evaluate_optuna),
            ]

            for method_name, runner in method_runners:
                with mlflow.start_run(run_name=f"{dataset_name}_{method_name}_fold_{outer_fold}", nested=True):
                    
                    if method_name == "optuna":
                        _, cv_mean, cv_std, test_acc, best_params, optuna_hist_df = runner(
                            X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
                            inner_cv=inner_cv, random_state=fold_seed,
                        )

                        hist_path = ARTIFACT_DIR / f"{dataset_name}_optuna_fold_{outer_fold}_history.csv"
                        optuna_hist_df.to_csv(hist_path, index=False)
                        mlflow.log_artifact(str(hist_path), artifact_path="optimization_history")
                    
                    else:
                        _, cv_mean, cv_std, test_acc, best_params = runner(
                            X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
                            inner_cv=inner_cv, random_state=fold_seed,
                        )

                    log_fold_metrics(
                        dataset_name=dataset_name, method_name=method_name,
                        outer_fold=outer_fold, cv_mean=cv_mean, cv_std=cv_std,
                        test_acc=test_acc, best_params=best_params,
                    )

                    results.append({
                        "dataset": dataset_name, "method": method_name, "outer_fold": outer_fold,
                        "cv_mean": cv_mean, "cv_std": cv_std, "test_acc": test_acc,
                        "best_params": json.dumps(best_params),
                    })

results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_PATH, index=False)
results_df